# Cited RAG over NIST AI RMF actions

Copyright (c) Patrick Hall (jphall@gwu.edu), 2026-27. Released under the MIT License; see the repository `SOFTWARE_LICENSE.txt`.

This notebook builds a small, inspectable retrieval-augmented generation pipeline. It ingests attributed source records, treats each AI RMF subcategory as one semantic chunk, compares lexical and embedding retrieval, uses an LLM judge to rerank a short candidate list, and generates an answer whose claims cite retrieved source IDs. 

## 1. Configure the OpenAI client

The client loads `OPENAI_API_KEY` from the project-root `.env`; if it is unavailable, the notebook asks for it without echoing the value.

In [1]:
# 1. Configure the OpenAI client
import csv
import json
import math
import os
import re
from collections import Counter
from getpass import getpass
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

# Search upward so the notebook works when launched from its folder or the repository root.
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env').exists()), Path.cwd())
load_dotenv(project_root / '.env')
api_key = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key: ')
client = OpenAI(api_key=api_key)
EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
TEXT_MODEL = os.getenv('OPENAI_TEXT_MODEL', 'gpt-5-nano')
print(f'Configured text model: {TEXT_MODEL}; embedding model: {EMBEDDING_MODEL}')

Configured text model: gpt-5-nano; embedding model: text-embedding-3-small


## 2. Ingest an attributed source corpus

Each record has a stable source ID, NIST AI RMF function, text, and a Playbook URL that leads to its relevant function or subcategory. Keeping attribution with the text is what later makes citation possible.

In [2]:
# 2. Ingest an attributed source corpus
corpus_path = project_root / 'lecture' / 'lec_5' / 'data' / 'rmf_rag_corpus.csv'
with corpus_path.open(newline='', encoding='utf-8') as handle:
    documents = list(csv.DictReader(handle))

required_fields = {'source_id', 'function', 'text', 'url'}
if not documents or set(documents[0]) != required_fields:
    raise ValueError('The corpus must contain source_id, function, text, and url columns.')
print(f'Loaded {len(documents)} attributed source records from {corpus_path.name}.')
for document in documents[:3]:
    print(f"[{document['source_id']}] {document['function']}: {document['text'][:95]}...")

Loaded 11 attributed source records from rmf_rag_corpus.csv.
[GV-1] GOVERN: Policies, processes, procedures, and practices across the organization related to mapping, meas...
[GV-1.1] GOVERN: Legal and regulatory requirements involving AI are understood, managed, and documented....
[GV-1.2-001] GOVERN: Connect new generative-AI policies, procedures, and processes to existing model, data, and IT g...


## 3. Use subcategories as semantic chunks

Each row in this corpus is already one AI RMF subcategory: an author-defined, meaningful unit with a stable ID and URL. For this small, structured corpus, use each subcategory as one semantic chunk rather than splitting it further.

In [3]:
# 3. Use subcategories as semantic chunks
# This is O(n): create exactly one chunk for each of the n source records.
chunks = [
    {**document, 'chunk_id': document['source_id'], 'chunk_text': document['text']}
    for document in documents
]
if len(chunks) != len(documents):
    raise RuntimeError('Expected exactly one chunk per source record.')
print(f'Used {len(chunks)} AI RMF subcategories as semantic chunks.')
for chunk in chunks[:3]:
    print(f"[{chunk['chunk_id']}] {len(chunk['chunk_text'].split()):2d} words | {chunk['chunk_text']}")

Used 11 AI RMF subcategories as semantic chunks.
[GV-1] 23 words | Policies, processes, procedures, and practices across the organization related to mapping, measuring, and managing AI risks are in place, transparent, and implemented effectively.
[GV-1.1] 11 words | Legal and regulatory requirements involving AI are understood, managed, and documented.
[GV-1.2-001] 21 words | Connect new generative-AI policies, procedures, and processes to existing model, data, and IT governance and to legal, compliance, and risk functions.


## 4. Define a question and build lexical retrieval

BM25 rewards query terms that occur in a chunk while discounting common terms and very long chunks. It is especially useful for exact terms and identifiers.

In [4]:
# 4. Define a question and build lexical retrieval
question = 'What should an organization document and monitor to manage AI risks after deployment?'

def tokenize(text):
    # Tokenization is O(c) in the number of characters and is intentionally visible for teaching.
    return re.findall(r"[a-z0-9]+", text.lower())

chunk_tokens = [tokenize(chunk['chunk_text']) for chunk in chunks]
document_frequency = Counter(token for tokens in chunk_tokens for token in set(tokens))
average_length = np.mean([len(tokens) for tokens in chunk_tokens])

def bm25_score(query, tokens, k1=1.2, b=0.75):
    """Return a transparent BM25 score for one query--chunk pair."""
    term_counts = Counter(tokens)
    score = 0.0
    # This loop is O(|q|); it visits each unique query term once.
    for term in set(tokenize(query)):
        frequency = term_counts[term]
        if not frequency:
            continue
        inverse_frequency = math.log(1 + (len(chunks) - document_frequency[term] + 0.5) / (document_frequency[term] + 0.5))
        length_adjustment = frequency + k1 * (1 - b + b * len(tokens) / average_length)
        score += inverse_frequency * frequency * (k1 + 1) / length_adjustment
    return score

lexical_scores = np.array([bm25_score(question, tokens) for tokens in chunk_tokens])
print(question)
for index in np.argsort(lexical_scores)[::-1][:3]:
    print(f"BM25={lexical_scores[index]:.3f} [{chunks[index]['chunk_id']}] {chunks[index]['chunk_text']}")

What should an organization document and monitor to manage AI risks after deployment?
BM25=5.006 [MG-2.1] Consider resources required to manage AI risks, together with viable non-AI alternatives, to reduce the magnitude or likelihood of potential impacts.
BM25=4.579 [GV-1] Policies, processes, procedures, and practices across the organization related to mapping, measuring, and managing AI risks are in place, transparent, and implemented effectively.
BM25=3.815 [MG-1.1] Determine whether the AI system achieves its intended purposes and stated objectives and whether its development or deployment should proceed.


## 5. Embed chunks and retrieve semantically

Embeddings make query and chunk texts comparable as vectors. The code normalizes vectors, so their dot product is cosine similarity.

In [5]:
# 5. Embed chunks and retrieve semantically
embedding_inputs = [chunk['chunk_text'] for chunk in chunks] + [question]
# The API call is O(nD) in the number of texts n and returned dimensions D; local normalization has the same order.
response = client.embeddings.create(model=EMBEDDING_MODEL, input=embedding_inputs, dimensions=512)
vectors = np.asarray([item.embedding for item in response.data], dtype=float)
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
chunk_vectors, query_vector = vectors[:-1], vectors[-1]
dense_scores = chunk_vectors @ query_vector

for index in np.argsort(dense_scores)[::-1][:3]:
    print(f"cosine={dense_scores[index]:.3f} [{chunks[index]['chunk_id']}] {chunks[index]['chunk_text']}")

cosine=0.778 [MS-3.1] Approaches, personnel, and documentation regularly identify and track existing, unanticipated, and emergent AI risks using intended and actual performance in deployed contexts.
cosine=0.732 [GV-1] Policies, processes, procedures, and practices across the organization related to mapping, measuring, and managing AI risks are in place, transparent, and implemented effectively.
cosine=0.707 [MG-4.1] Post-deployment AI-system monitoring plans include mechanisms for user and stakeholder input, appeal and override, decommissioning, incident response, recovery, and change management.


## 6. Fuse lexical and dense rankings

Reciprocal-rank fusion combines positions rather than incomparable raw score scales. A candidate that ranks well in either retrieval mode can survive to reranking.

In [6]:
# 6. Fuse lexical and dense rankings
def reciprocal_rank_fusion(*score_arrays, constant=60):
    """Combine rank lists; sorting each list costs O(n log n)."""
    # Start each chunk at zero; this avoids comparing incompatible raw scores.
    fused = np.zeros(len(score_arrays[0]))
    for scores in score_arrays:
        # Sort this retriever's chunks from best score to worst score.
        for rank, index in enumerate(np.argsort(scores)[::-1], start=1):
            # Add rank-based credit: rank 1 earns 1/(60 + 1), rank 2 earns less.
            # The constant dampens the difference between nearby ranks.
            fused[index] += 1 / (constant + rank)
    return fused

# A chunk ranks well when one or both retrievers place it near the top.
hybrid_scores = reciprocal_rank_fusion(lexical_scores, dense_scores)
# Keep the six highest fused ranks for the more expensive reranking stage.
candidate_indices = np.argsort(hybrid_scores)[::-1][:6]
# Retain component scores so the hybrid decision remains inspectable.
candidates = [chunks[index] | {'bm25': float(lexical_scores[index]), 'cosine': float(dense_scores[index]), 'hybrid': float(hybrid_scores[index])} for index in candidate_indices]
for candidate in candidates:
    print(f"[{candidate['chunk_id']}] hybrid={candidate['hybrid']:.4f}; BM25={candidate['bm25']:.3f}; cosine={candidate['cosine']:.3f}")

[GV-1] hybrid=0.0323; BM25=4.579; cosine=0.732
[MG-2.1] hybrid=0.0320; BM25=5.006; cosine=0.693
[MS-3.1] hybrid=0.0311; BM25=1.420; cosine=0.778
[MG-1.1] hybrid=0.0308; BM25=3.815; cosine=0.601
[MG-4.1] hybrid=0.0308; BM25=1.622; cosine=0.707
[MS-1.1] hybrid=0.0303; BM25=1.729; cosine=0.664


## 7. Use an LLM judge to rerank candidates

The LLM acts as a judge: it reads the question and each retrieved candidate, then selects the three most relevant chunks. Its structured response makes that judgment inspectable. It scores relevance only; it does not decide whether the final answer is supported.

In [7]:
# 7. Use an LLM judge to rerank candidates
# The LLM judges relevance for six retrieved chunks; it does not generate the answer yet.
rerank_schema = {
    'type': 'object',
    'properties': {
        'ranked_chunk_ids': {'type': 'array', 'items': {'type': 'string'}, 'minItems': 3, 'maxItems': 3},
        'rationale': {'type': 'string'},
    },
    'required': ['ranked_chunk_ids', 'rationale'],
    'additionalProperties': False,
}
candidate_text = '\n\n'.join(f"[{item['chunk_id']}] {item['chunk_text']}" for item in candidates)
# Submit the question and candidates to the LLM judge for a structured ranking.
rerank_response = client.responses.create(
    model=TEXT_MODEL,
    instructions='Rank exactly three candidate chunks by relevance to the question. Return only IDs supplied in the candidate set. Prefer passages that directly answer the question.',
    input=f'Question: {question}\n\nCandidates:\n{candidate_text}',
    max_output_tokens=800,
    reasoning={'effort': 'low'},
    text={'format': {'type': 'json_schema', 'name': 'rag_rerank', 'strict': True, 'schema': rerank_schema}},
    store=False,
)
if not rerank_response.output_text:
    raise RuntimeError(f'Reranker returned no visible JSON; status={rerank_response.status}.')
rerank_result = json.loads(rerank_response.output_text)
candidate_by_id = {item['chunk_id']: item for item in candidates}
if any(chunk_id not in candidate_by_id for chunk_id in rerank_result['ranked_chunk_ids']):
    raise ValueError('The reranker returned an ID outside the candidate set.')
# Keep only the three chunks the LLM judge selected for answer generation.
selected = [candidate_by_id[chunk_id] for chunk_id in rerank_result['ranked_chunk_ids']]
print('Selected chunks:', rerank_result['ranked_chunk_ids'])
print('Reranker rationale:', rerank_result['rationale'])

Selected chunks: ['MG-4.1', 'MS-3.1', 'GV-1']
Reranker rationale: MG-4.1 focuses on post-deployment monitoring plans and the included mechanisms. MS-3.1 covers ongoing identification and tracking of AI risks in deployed contexts. GV-1 provides the organizational policy context for mapping, measuring, and managing AI risks, which underpins post-deployment governance.


## 8. Generate a bounded answer with citations

The answer contract requires source IDs for material claims and a short abstention when the selected context is insufficient. The final checks make missing or invented citations visible.

In [8]:
# 8. Generate a bounded answer with citations
context = '\n\n'.join(f"[{item['chunk_id']}] {item['chunk_text']}" for item in selected)
answer_response = client.responses.create(
    model=TEXT_MODEL,
    instructions='Answer only from the supplied context. Cite every substantive claim with one or more bracketed chunk IDs exactly as supplied. If the context is insufficient, say so instead of adding facts. Keep the answer under 150 words.',
    input=f'Question: {question}\n\nContext:\n{context}',
    max_output_tokens=1000,
    reasoning={'effort': 'low'},
    store=False,
)
answer = answer_response.output_text.strip()
if not answer:
    raise RuntimeError(f'Generator returned no visible answer; status={answer_response.status}.')

cited_ids = set(re.findall(r'\[([^\]]+)\]', answer))
selected_ids = {item['chunk_id'] for item in selected}
unsupported_ids = cited_ids - selected_ids
if unsupported_ids:
    raise ValueError(f'Answer used unsupported citation IDs: {sorted(unsupported_ids)}')
if not cited_ids:
    raise ValueError('Answer contains no citations; inspect the generator output.')

print(answer)
print('\nRetrieved sources:')
for item in selected:
    print(f"[{item['chunk_id']}] {item['source_id']} | {item['url']}")

- Post-deployment monitoring plans: document and monitor mechanisms for user and stakeholder input, appeal and override, decommissioning, incident response, recovery, and change management. [MG-4.1]

- Risk identification and tracking: regularly identify and track existing, unanticipated, and emergent AI risks using both intended and actual performance in deployed contexts. [MS-3.1]

- Organizational risk governance: ensure policies, processes, procedures, and practices related to mapping, measuring, and managing AI risks are in place, transparent, and effectively implemented. [GV-1]

Retrieved sources:
[MG-4.1] MG-4.1 | https://airc.nist.gov/airmf-resources/playbook/manage/#button-MANAGE%204.1
[MS-3.1] MS-3.1 | https://airc.nist.gov/airmf-resources/playbook/measure/#button-MEASURE%203.1
[GV-1] GV-1 | https://airc.nist.gov/airmf-resources/playbook/govern/


This small corpus makes the pipeline legible but does not establish retrieval quality for a real deployment. Extend it with an authorized, versioned corpus; access controls before retrieval; a held-out question set with expected source IDs; citation-entailment review; latency and cost measurement; and monitoring for corpus or query drift.